<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/BaselineModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q datasets

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from datasets import load_dataset

import numpy as np

img_size = 150
batch_size = 50
num_classes = 4

In [14]:
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5120
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1280
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented'])}


In [15]:
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]

def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)   # (H,W) -> (H,W,1)

    def keep_same():
        return images                            # already (H,W,C)

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)


def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        # (H,W,1) -> (H,W,3)
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        # (H,W,4) -> (H,W,3)
        return images[..., :3]

    def keep_same():
        return images

    # 1-channel -> 3-channel
    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)

    new_channels = tf.shape(images)[-1]

    # 4-channel (e.g., RGBA) -> drop alpha
    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_same)
    return images   # guaranteed (H,W,3) for our dataset


# simple rescale-only preprocess to mimic paper’s rescaling step
def simple_preprocess(x):
    return x / 255.0

# ---------- generic HF -> tf.data wrapper (like your to_tensorflow_dataset) ----------

def to_tensorflow_dataset(dataset_split, image_size, preprocess_fn, shuffle=False):
    dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )

    def preprocess_input(example_dict):
        images = tf.cast(example_dict["image"], tf.float32)
        images = ensure_channel_dim(images)                 # add channel dim if needed
        images = ensure_rgb_channels(images)                # force 3 channels
        images.set_shape([None, None, 3])                   # static shape: (H,W,3)
        images = tf.image.resize(images, (image_size, image_size))
        images = preprocess_fn(images)                      # here: rescale to [0,1]

        labels = tf.cast(example_dict["label"], tf.int32)
        labels = tf.one_hot(labels, depth=num_classes)      # for categorical CE
        return images, labels

    dataset_tf = dataset_tf.map(preprocess_input,
                                num_parallel_calls=tf.data.AUTOTUNE)
    dataset_tf = dataset_tf.batch(batch_size)
    return dataset_tf.prefetch(tf.data.AUTOTUNE)

# ---------- build your 150x150x3 datasets ----------

train_ds = to_tensorflow_dataset(train, img_size, simple_preprocess, shuffle=True)
val_ds   = to_tensorflow_dataset(val,   img_size, simple_preprocess, shuffle=False)
test_ds  = to_tensorflow_dataset(test,  img_size, simple_preprocess, shuffle=False)

In [16]:
model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),          # 1

    layers.Conv2D(16, (3, 3), activation='relu'),  # 2
    layers.MaxPooling2D((2, 2)),                  # 3

    layers.Conv2D(32, (3, 3), activation='relu'),  # 4
    layers.MaxPooling2D((2, 2)),                  # 5

    layers.Flatten(),                             # 6
    layers.Dense(154, activation='relu'),         # 7
    layers.Dense(num_classes, activation='softmax')
])

model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 154)            │     6,386,842 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │           620 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,392,550 (24.39 MB)

 Trainable params: 6,392,550 (24.39 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
opt = optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[reduce_lr]
)

test_loss, test_acc = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)


Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 91s 1s/step - accuracy: 0.5068 - loss: 1.1535 - val_accuracy: 0.6133 - val_loss: 0.8891 - learning_rate: 0.0010
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.6413 - loss: 0.7849 - val_accuracy: 0.7051 - val_loss: 0.6667 - learning_rate: 0.0010
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 83s 1s/step - accuracy: 0.8210 - loss: 0.4682 - val_accuracy: 0.8008 - val_loss: 0.4785 - learning_rate: 0.0010
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 141s 999ms/step - accuracy: 0.8952 - loss: 0.2944 - val_accuracy: 0.8799 - val_loss: 0.3009 - learning_rate: 0.0010
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.9582 - loss: 0.1412 - val_accuracy: 0.9189 - val_loss: 0.2165 - learning_rate: 0.0010
Epoch 6/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.9846 - loss: 0.0715 - val_accuracy: 0.9189 - val_loss: 0.2128 - learning_rate: 0.0010
Epoch 7/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 83s 1s/step - accuracy: 0.9919 - loss: 0.0408 - val